In [1]:
from pathlib import Path
from datetime import date
import sys
import pandas as pd
import numpy as np
BASE_DIR = Path.cwd().parent.parent.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

RIDGE_ALPHA = 1.0                   # Ridge 규제 강도입니다. 가장 중요하게 조정할 값입니다. 값이 클수록 단어별 계수를 강하게 축소하여 과적합을 줄입니다.
RIDGE_FIT_INTERCEPT = True          # 절편을 학습할지 결정합니다. 목표값 평균이 정확히 0이라고 보장할 수 없으므로 기본적으로 True가 적절합니다.
RIDGE_SOLVER = "lsqr"               # 계수를 계산하는 방법입니다. "lsqr"는 현재처럼 특성 수가 많고 입력이 희소행렬인 경우에 사용할 수 있습니다.
RIDGE_TOL = 1e-4                    # 계산을 어느 정도 정밀도에서 종료할지 결정합니다. 작을수록 정밀하지만 시간이 더 걸릴 수 있습니다.
RIDGE_MAX_ITER = None               # 반복 계산의 최대 횟수입니다. None이면 solver의 기본값을 사용합니다.
RIDGE_POSITIVE = False              # 모든 계수를 양수로 제한할지 결정합니다. 부정적인 단어 효과도 표현해야 하므로 현재 모델에서는 False가 맞습니다.

NGRAM_RANGE = (1, 1)
MIN_DF = 5
MAX_DF = 0.95
MAX_FEATURES = 10_000
SUBLINEAR_TF = True
# BASE_DIR = Path(__file__).resolve().parents[2]
DATA_DIR = BASE_DIR / "data"
INPUT_DIR = DATA_DIR / "tokenized"
INPUT_PATTERN = "*.jsonl"

INPUT_PATHS = tuple(
    sorted(INPUT_DIR.glob(INPUT_PATTERN))
)
PRICE_PATH = DATA_DIR / "raw" / "price_api.jsonl"

VALIDATION_START_DATE = date(2026,7,10)

In [2]:
def split_labeled_dataset_by_date(
    dataset: pd.DataFrame,
    validation_start_date: date,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    지정한 날짜 이전은 훈련 데이터로,
    지정한 날짜부터는 검증 데이터로 분리한다.
    """
    train_df = dataset.loc[
        dataset["model_date"] < validation_start_date
    ].copy()

    validation_df = dataset.loc[
        dataset["model_date"] >= validation_start_date
    ].copy()

    if train_df.empty:
        raise ValueError("훈련 데이터가 비어 있습니다.")

    if validation_df.empty:
        raise ValueError("검증 데이터가 비어 있습니다.")

    return train_df, validation_df

In [ ]:
from pilos.analysis.daily_dataset import (
    build_labeled_daily_dataset,
    )

# 휴장일이 제거된 학습후보 데이터셋 생성
dataset = build_labeled_daily_dataset(
    input_paths=INPUT_PATHS,
    price_path=PRICE_PATH,
)

In [4]:
# 댓글수의 수치를 압축하기위한 로그화
dataset = dataset.assign(
    log_comment_count=np.log1p(
        dataset["comment_count"].astype(float)
    )
)
# 데이터셋 확인
dataset[
    [
        "stock_code",
        "model_date",
        "comment_count",
        "log_comment_count",
        "supply_demand_index",
    ]
].head()

,stock_code,model_date,comment_count,log_comment_count,supply_demand_index
0,005380,2026-06-01,1767,7.477604,0.0259
1,005490,2026-06-01,193,5.267858,-0.0009
2,005930,2026-06-01,10439,9.253400,-0.1119
3,034020,2026-06-01,1198,7.089243,0.0309
4,035420,2026-06-01,8030,8.991064,0.0293


In [5]:
# 기준 날짜확인
trading_dates = sorted(
    dataset["model_date"].unique()
)

split_index = int(
    len(trading_dates) * 0.75
)

candidate_validation_start_date = (
    trading_dates[split_index]
)

candidate_validation_start_date

datetime.date(2026, 7, 10)

In [6]:
# 훈련,검증 데이터 분리
train_df, validation_df = (
    split_labeled_dataset_by_date(
        dataset=dataset,
        validation_start_date=VALIDATION_START_DATE,
    )
)
print(f"훈련 행 수: {len(train_df):,}")
print(f"검증 행 수: {len(validation_df):,}")

print(
    "훈련 기간:",
    train_df["model_date"].min(),
    "~",
    train_df["model_date"].max(),
)

print(
    "검증 기간:",
    validation_df["model_date"].min(),
    "~",
    validation_df["model_date"].max(),
)
train_df["model_date"].max() < validation_df["model_date"].min()

훈련 행 수: 252
검증 행 수: 90
훈련 기간: 2026-06-01 ~ 2026-07-09
검증 기간: 2026-07-10 ~ 2026-07-24


True

In [ ]:
from pilos.analysis.vectorizer import (
    create_tfidf_vectorizer,
    vectorize_comments,
    transform_comments,
)
# 백터라이저 생성
vectorizer = create_tfidf_vectorizer(
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES,
    sublinear_tf=SUBLINEAR_TF,
)
# 훈련데이터 백터화
train_tfidf = vectorize_comments(
    documents=train_df["tfidf_text"],
    vectorizer=vectorizer,
)
# 검증 문서는 훈련 데이터로 학습한 Vectorizer를 변경하지 않고 변환만 한다.
validation_tfidf = transform_comments(
    documents=validation_df["tfidf_text"],
    vectorizer=vectorizer,
)
print("훈련 TF-IDF:", train_tfidf.shape)
print("검증 TF-IDF:", validation_tfidf.shape)

훈련 TF-IDF: (252, 10000)
검증 TF-IDF: (90, 10000)


In [8]:
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
# 스케일러소환
comment_count_scaler = StandardScaler()

# Ridge는 특성값이 아니라 모델 계수의 크기를 규제한다.
# 특성들의 숫자 크기가 다르면 같은 예측 효과에 필요한 계수 크기도 달라져 규제가 특성마다 불균형하게 작동할 수 있다.
# 따라서 로그 댓글 수를 평균 0, 표준편차 1로 표준화한다.


# 학습데이터에 평균,표준편차등을 파악해 스켈일링한다
train_comment_count = (
    comment_count_scaler.fit_transform(
        train_df[["log_comment_count"]]
    )
)
# 학습데이터로 저장된 계산식으로 검증데이터를 스켈일링한다
validation_comment_count = (
    comment_count_scaler.transform(
        validation_df[["log_comment_count"]]
    )
)
# tfidf는 희소행렬을 반환한다 따라서 스켈링링한 댓글수를 희소행렬로 변환하여 열방향으로 연결한다
# 이때 Ridge모델이 효울적으로 사용할수있는 csr 희소행렬형식으로 변환한다
train_features = hstack(
    [
        train_tfidf,
        csr_matrix(train_comment_count),
    ],
    format="csr",
)

validation_features = hstack(
    [
        validation_tfidf,
        csr_matrix(validation_comment_count),
    ],
    format="csr",
)

In [ ]:
# 모델에 Y(라벨) : 정답, 목표값을 생성한다
# 이떄 모델이 입력받을수있게 시리즈를 1차원 넘파이배열로 변환한다
train_target = train_df[
    "supply_demand_index"
].to_numpy(
    dtype=float
)

validation_target = validation_df[
    "supply_demand_index"
].to_numpy(
    dtype=float
)

print("훈련 특성:", train_features.shape)
print("훈련 목표:", train_target.shape)

print("검증 특성:", validation_features.shape)
print("검증 목표:", validation_target.shape)

훈련 특성: (252, 10001)
훈련 목표: (252,)
검증 특성: (90, 10001)
검증 목표: (90,)


In [10]:
from pilos.analysis.ridge_model import (
    create_ridge_model,
    fit_ridge_model,
    predict_ridge_model,
)

# 릿지모델생성
ridge_model = create_ridge_model(
    alpha=RIDGE_ALPHA,
    fit_intercept=RIDGE_FIT_INTERCEPT,
    solver=RIDGE_SOLVER,
    tol=RIDGE_TOL,
    max_iter=RIDGE_MAX_ITER,
    positive=RIDGE_POSITIVE,
)
# 훈련데이터로 모델학습
fit_ridge_model(
    model=ridge_model,
    features=train_features,
    target=train_target,
)
# 훈련데이터 예측
train_predictions = predict_ridge_model(
    model=ridge_model,
    features=train_features,
)
# 검증데이터예측
validation_predictions = predict_ridge_model(
    model=ridge_model,
    features=validation_features,
)

print("훈련 예측:", train_predictions.shape)
print("검증 예측:", validation_predictions.shape)

훈련 예측: (252,)
검증 예측: (90,)


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def calculate_regression_metrics(
    target: np.ndarray,
    predictions: np.ndarray,
) -> dict[str, float]:
    """회귀 모델의 MAE, RMSE, R²를 계산한다."""
    # MAE : 각 행의 오차 절댓값을 구한 뒤 평균을 계산한다
    # 오차 = 실제값 - 예측값 MAE = |오차|의 평균
    # 오차의 방향은 제거하며 예측값이 실제값에서 얼마나 벗어났는지 수치화한다
    mae = mean_absolute_error(
        target,
        predictions,
    )
    # RMSE : 오차를 제곱하여 평균을 계산한후 루트를 씌운다
    # 오차 = 실제값 - 예측값 RMSE =  오차^2의평균의 제곱근
    # 큰오차에 더강한 벌점을 주도록 수치화한다
    rmse = np.sqrt(
        mean_squared_error(
            target,
            predictions,
        )
    )
    # R2 : 오차의제곱을 예측값의 평균의 제곱으로 나누것을 1에서 뺀다
    # 오차 = 실제값 - 예측값 R² = 1 - 모델의 제곱오차 합 / 평균 예측의 제곱오차 합
    # 1이면 완벽한예측 
    # 0이면 실제값 평균만 예측한 것과 같은 수준
    # 음수면 실제값 평균만 예측한 것도받 성능이 낮다
    r2 = r2_score(
        target,
        predictions,
    )

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }

In [12]:
train_metrics = calculate_regression_metrics(
    target=train_target,
    predictions=train_predictions,
)

validation_metrics = calculate_regression_metrics(
    target=validation_target,
    predictions=validation_predictions,
)

print("훈련:", train_metrics)
print("검증:", validation_metrics)

훈련: {'mae': 0.1037654736762689, 'rmse': 0.12862690023649995, 'r2': 0.7095286236000256}
검증: {'mae': 0.18160011110668905, 'rmse': 0.2214389629288556, 'r2': 0.07565646296747142}


In [ ]:
from pilos.analysis.ridge_model import (
    get_ridge_coefficients,
    get_ridge_intercept,
)
from pilos.analysis.vectorizer import (
    get_feature_names,
)

# 가중치를 통한 단어 추출 준비과정
coefficients = get_ridge_coefficients(
    ridge_model
)

intercept = get_ridge_intercept(
    ridge_model
)

text_feature_names = get_feature_names(
    vectorizer
)

print("전체 모델 계수:", coefficients.shape)
print("텍스트 특성 이름:", text_feature_names.shape)
print("절편:", intercept)

전체 모델 계수: (10001,)
텍스트 특성 이름: (10000,)
절편: -0.003559471259900908


In [14]:
# 댓글수 열을 자르는작업
text_coefficients = coefficients[
    :len(text_feature_names)
]

comment_count_coefficient = coefficients[
    len(text_feature_names)
]

In [15]:
# 특성의 이름과 모델 가중치를 연결
feature_weight_df = pd.DataFrame(
    {
        "feature": text_feature_names,
        "coefficient": text_coefficients,
    }
)

In [ ]:
# 양의가중치 키워드 20위 추출
top_positive_features = (
    feature_weight_df
    .sort_values(
        "coefficient",
        ascending=False,
    )
    .head(20)
)

top_positive_features

,feature,coefficient
7423,전지,0.217998
3815,배터리,0.141247
8244,차,0.133124
7840,줍줍,0.121536
9387,하락,0.111770
2172,답,0.106506
162,lg,0.100998
3546,미치,0.100451
1786,내려가,0.099762
3846,버리,0.093705


In [ ]:
# 음의가중치 키워드 20위 추출
top_negative_features = (
    feature_weight_df
    .sort_values(
        "coefficient",
        ascending=True,
    )
    .head(20)
)

top_negative_features

,feature,coefficient
4878,셀트리온,-0.159845
4599,상,-0.127998
2737,뚫,-0.106698
6152,연기금,-0.104126
504,감흥,-0.102567
4642,상처,-0.098167
2127,단주,-0.097686
2542,드디어,-0.096925
5016,수급,-0.095545
4660,상한가,-0.089279


In [18]:
print(
    "표준화된 로그 댓글 수 계수:",
    comment_count_coefficient,
)

표준화된 로그 댓글 수 계수: 0.050072959498867226


In [ ]:
# 특정 종목 날짜의 예측 기여 키워드 추출
# 샘플 댓글 선정
row_index = 0

sample_metadata = validation_df.iloc[
    row_index
]

sample_tfidf = validation_tfidf.getrow(
    row_index
)

# TFIDF점수가 0이아닌 특성추출
feature_indices = sample_tfidf.indices
tfidf_values = sample_tfidf.data
# 예측 기여도 계산
text_contributions = (
    tfidf_values
    * text_coefficients[feature_indices]
)
# 검토용 df생성
contribution_df = pd.DataFrame(
    {
        "feature": text_feature_names[
            feature_indices
        ],
        "tfidf": tfidf_values,
        "coefficient": text_coefficients[
            feature_indices
        ],
        "contribution": text_contributions,
    }
)


,feature,tfidf,coefficient,contribution
886,차,0.053116,0.133124,0.007071
852,줍줍,0.042203,0.121536,0.005129
183,내려가,0.037435,0.099762,0.003735
184,내리,0.039666,0.090375,0.003585
1013,하락,0.031357,0.111770,0.003505
294,또,0.042085,0.080070,0.003370
518,생각,0.039091,0.080159,0.003133
224,다시,0.037958,0.080417,0.003052
1069,환불,0.038359,0.070673,0.002711
71,공매도,0.034817,0.076506,0.002664


In [22]:
# 예측을 상승시키는 키워드
top_positive_contributions = (
    contribution_df
    .sort_values(
        "contribution",
        ascending=False,
    )
    .head(10)
)

top_positive_contributions

,feature,tfidf,coefficient,contribution
886,차,0.053116,0.133124,0.007071
852,줍줍,0.042203,0.121536,0.005129
183,내려가,0.037435,0.099762,0.003735
184,내리,0.039666,0.090375,0.003585
1013,하락,0.031357,0.111770,0.003505
294,또,0.042085,0.080070,0.003370
518,생각,0.039091,0.080159,0.003133
224,다시,0.037958,0.080417,0.003052
1069,환불,0.038359,0.070673,0.002711
71,공매도,0.034817,0.076506,0.002664


In [23]:
# 예측을 하락시키는 키워드
top_negative_contributions = (
    contribution_df
    .sort_values(
        "contribution",
        ascending=True,
    )
    .head(10)
)

top_negative_contributions

,feature,tfidf,coefficient,contribution
505,상,0.045034,-0.127998,-0.005764
351,멀,0.049196,-0.085865,-0.004224
297,뚫,0.037553,-0.106698,-0.004007
268,드디어,0.040657,-0.096925,-0.003941
599,쏘,0.044536,-0.085112,-0.003791
492,사이드,0.054864,-0.061061,-0.003350
551,수급,0.032646,-0.095545,-0.003119
627,알림,0.051801,-0.059713,-0.003093
1065,화이팅,0.034817,-0.086829,-0.003023
511,상한가,0.033275,-0.089279,-0.002971


In [25]:
# 댓글 활동량 기여도
comment_count_contribution = (
    validation_comment_count[row_index, 0]
    * comment_count_coefficient
)
comment_count_contribution

np.float64(0.02463085677291122)

In [26]:
reconstructed_prediction = (
    intercept
    + text_contributions.sum()
    + comment_count_contribution
)

print(
    "모델 예측:",
    validation_predictions[row_index],
)

print(
    "기여도 합으로 재구성:",
    reconstructed_prediction,
)

모델 예측: 0.012530230946488558
기여도 합으로 재구성: 0.012530230946488571


In [31]:
positive_keywords = (
    top_positive_contributions[
        [
            "feature",
            "contribution",
        ]
    ]
    .rename(
        columns={
            "feature": "keyword",
        }
    )
    .to_dict(
        orient="records"
    )
)

negative_keywords = (
    top_negative_contributions[
        [
            "feature",
            "contribution",
        ]
    ]
    .rename(
        columns={
            "feature": "keyword",
        }
    )
    .to_dict(
        orient="records"
    )
)
actual_index = float(
    validation_target[row_index]
)

predicted_index = float(
    validation_predictions[row_index]
)

prediction_error = (
    actual_index
    - predicted_index
)

In [32]:
prediction_result = {
    "stock_code": str(
        sample_metadata["stock_code"]
    ),
    "model_date": sample_metadata[
        "model_date"
    ].isoformat(),
    "actual_index": actual_index,
    "predicted_index": predicted_index,
    "prediction_error": float(
        prediction_error
    ),
    "comment_count": int(
        sample_metadata["comment_count"]
    ),
    "comment_count_contribution": float(
        comment_count_contribution
    ),
    "positive_keywords": positive_keywords,
    "negative_keywords": negative_keywords,
}
prediction_result

{'stock_code': '005380',
 'model_date': '2026-07-10',
 'actual_index': -0.2569,
 'predicted_index': 0.012530230946488558,
 'prediction_error': -0.26943023094648855,
 'comment_count': 909,
 'comment_count_contribution': 0.02463085677291122,
 'positive_keywords': [{'keyword': '차', 'contribution': 0.0070710537225027415},
  {'keyword': '줍줍', 'contribution': 0.005129164497784571},
  {'keyword': '내려가', 'contribution': 0.003734549213472931},
  {'keyword': '내리', 'contribution': 0.003584800360670124},
  {'keyword': '하락', 'contribution': 0.0035047536852044698},
  {'keyword': '또', 'contribution': 0.0033697573846607947},
  {'keyword': '생각', 'contribution': 0.0031334420945782937},
  {'keyword': '다시', 'contribution': 0.003052485761638427},
  {'keyword': '환불', 'contribution': 0.002710964219525282},
  {'keyword': '공매도', 'contribution': 0.002663707895018377}],
 'negative_keywords': [{'keyword': '상', 'contribution': -0.005764225894771738},
  {'keyword': '멀', 'contribution': -0.00422421494740773},
  {'ke